## Visualize verification

### Environment Setup

In [32]:
# Import package
import os
import sys
import numpy as np

from glob import glob
from typing import Tuple
from pathlib import Path

from matplotlib import pyplot as plt

sys.path.append("/Users/joseph/Desktop/NTU Course/114-2/Data Assimilation/Project/Code/src")
import verification as ver #type: ignore

# Path setup
root_path: Path = Path("/Users/joseph/Desktop/NTU Course/114-2/Data Assimilation/Project")
file_path: Path = root_path / "Files"
fig_path : Path = root_path / "Figure"

plt.style.use(root_path / "Code" / "src" / "stylelist.mplstyle")


### Load data

In [33]:
# load nature run
nature_run: dict[str, np.ndarray] = dict(np.load(file_path / "traj" / "nature_run.npz"))

# load free run
free_run: dict[str, np.ndarray] = dict(np.load(file_path / "traj" / "free_run.npz"))

# load OI
OI_traj: dict[str, dict[str, np.ndarray]] = {
    "on_X": dict(np.load(file_path / "traj" / "OI" / "NMC" / "on_X.npz")),
    "on_Y": dict(np.load(file_path / "traj" / "OI" / "NMC" / "on_Y.npz")),
    "on_Z": dict(np.load(file_path / "traj" / "OI" / "NMC" / "on_Z.npz")),
    "on_XYZ": dict(np.load(file_path / "traj" / "OI" / "NMC" / "on_X-Y-Z.npz")),
}

# load 3DVar
normal_3DVar_traj: dict[str, dict[str, np.ndarray]] = {
    "on_X": dict(np.load(file_path / "traj" / "3DVar" / "NMC" / "on_X.npz")),
    "on_Y": dict(np.load(file_path / "traj" / "3DVar" / "NMC" / "on_Y.npz")),
    "on_Z": dict(np.load(file_path / "traj" / "3DVar" / "NMC" / "on_Z.npz")),
    "on_XYZ": dict(np.load(file_path / "traj" / "3DVar" / "NMC" / "on_X-Y-Z.npz")),
}

# load incremental 3DVar
incremental_3DVar_traj: dict[str, dict[str, np.ndarray]] = {
    "on_X": dict(np.load(file_path / "traj" / "3DVar" / "NMC" / "on_X.npz")),
    "on_Y": dict(np.load(file_path / "traj" / "3DVar" / "NMC" / "on_Y.npz")),
    "on_Z": dict(np.load(file_path / "traj" / "3DVar" / "NMC" / "on_Z.npz")),
    "on_XYZ": dict(np.load(file_path / "traj" / "3DVar" / "NMC" / "on_X-Y-Z.npz")),
}

### Calculate verification

#### Calculate RMSE

In [34]:
# free run
free_rmse: Tuple[np.ndarray, ...] = ver.calc_RMSE(
    truth = (nature_run["X"], nature_run["Y"], nature_run["Z"]),
    estim = (free_run["X"], free_run["Y"], free_run["Z"]),
)

# OI
OI_rmse: dict[str, Tuple[np.ndarray, ...]] = {
    key: ver.calc_RMSE(
        truth = (nature_run["X"], nature_run["Y"], nature_run["Z"]),
        estim = (OI_traj[key]["X"], OI_traj[key]["Y"], OI_traj[key]["Z"])
    )
    for key in OI_traj.keys()
}

# 3DVar
n3DVar_rmse: dict[str, Tuple[np.ndarray, ...]] = {
    key: ver.calc_RMSE(
        truth = (nature_run["X"], nature_run["Y"], nature_run["Z"]),
        estim = (normal_3DVar_traj[key]["X"], normal_3DVar_traj[key]["Y"], normal_3DVar_traj[key]["Z"])
    )
    for key in normal_3DVar_traj.keys()
}

# incremental 3DVar
i3DVar_rmse: dict[str, Tuple[np.ndarray, ...]] = {
    key: ver.calc_RMSE(
        truth = (nature_run["X"], nature_run["Y"], nature_run["Z"]),
        estim = (incremental_3DVar_traj[key]["X"], incremental_3DVar_traj[key]["Y"], incremental_3DVar_traj[key]["Z"])
    )
    for key in incremental_3DVar_traj.keys()
}


#### Calculate ACC

In [35]:
# free run
free_acc: Tuple[np.ndarray, ...] = ver.calc_ACC(
    truth = (nature_run["X"], nature_run["Y"], nature_run["Z"]),
    estim = (free_run["X"], free_run["Y"], free_run["Z"]),
    free  = (free_run["X"], free_run["Y"], free_run["Z"]),
)

# OI
OI_acc: dict[str, Tuple[np.ndarray, ...]] = {
    key: ver.calc_ACC(
        truth = (nature_run["X"], nature_run["Y"], nature_run["Z"]),
        estim = (OI_traj[key]["X"], OI_traj[key]["Y"], OI_traj[key]["Z"]),
        free  = (free_run["X"], free_run["Y"], free_run["Z"]),
    )
    for key in OI_traj.keys()
}

# 3DVar
n3DVar_acc: dict[str, Tuple[np.ndarray, ...]] = {
    key: ver.calc_ACC(
        truth = (nature_run["X"], nature_run["Y"], nature_run["Z"]),
        estim = (normal_3DVar_traj[key]["X"], normal_3DVar_traj[key]["Y"], normal_3DVar_traj[key]["Z"]),
        free  = (free_run["X"], free_run["Y"], free_run["Z"]),
    )
    for key in normal_3DVar_traj.keys()
}

# incremental 3DVar
i3DVar_acc: dict[str, Tuple[np.ndarray, ...]] = {
    key: ver.calc_ACC(
        truth = (nature_run["X"], nature_run["Y"], nature_run["Z"]),
        estim = (incremental_3DVar_traj[key]["X"], incremental_3DVar_traj[key]["Y"], incremental_3DVar_traj[key]["Z"]),
        free  = (free_run["X"], free_run["Y"], free_run["Z"]),
    )
    for key in incremental_3DVar_traj.keys()
}


### Visualize

#### Visualize RMSE

##### Across different OI scale

In [36]:
# coordinate setting
time_lim: int = 2000

skips: int = 40

time: np.ndarray = np.linspace(0, int(time_lim*0.01), time_lim)[::skips]


# 1. 將圖表拉寬，讓 X 軸的時間展開，減少線條在垂直方向的擁擠感
fig, ax = plt.subplots(3, figsize=(10, 8), sharex="col") 

# 2. 建立一個統一樣式的字典，精準控制每條線的視覺層級
style_dict = {
    "free": {"color": "gray", "linestyle": "--", "linewidth": 2.5, "alpha": 0.8, "zorder": 1},
    "on_X": {"color": "tab:orange", "linestyle": "-", "linewidth": 2.0, "alpha": 0.8, "zorder": 2},
    "on_Y": {"color": "tab:green", "linestyle": "-", "linewidth": 2.0, "alpha": 0.8, "zorder": 2},
    "on_Z": {"color": "tab:red", "linestyle": "-", "linewidth": 2.0, "alpha": 0.8, "zorder": 2},
    "on_XYZ": {"color": "tab:purple", "linestyle": "-", "linewidth": 3.5, "alpha": 1.0, "zorder": 5} # 最粗、最不透明、放在最上層
}

for i in range(3):
    # 畫 Free Run (黑色虛線墊底)
    ax[i].semilogy(
        time, free_rmse[i][:time_lim][::skips],
        label="Free Run", **style_dict["free"]
    )
    
    # 畫出各種同化設定
    for (key, value) in OI_rmse.items():
        # 如果是該子圖的主角（例如在 i=0 的圖畫 on_X），可以讓它稍微粗一點，以突顯 Downscale control
        lw = 3.0 if (i==0 and key=="on_X") or (i==1 and key=="on_Y") or (i==2 and key=="on_Z") else style_dict[key]["linewidth"]
        
        ax[i].semilogy(
            time, value[i][:time_lim][::skips],
            label=key.replace("_", " "), # 把底線換成空格比較好看
            color=style_dict[key]["color"],
            linestyle=style_dict[key]["linestyle"],
            linewidth=lw,
            alpha=style_dict[key]["alpha"],
            zorder=style_dict[key]["zorder"]
        )

    ax[i].set_xlim(0, 20)
    # 加上網格線，幫助投影幕上對齊數值
    ax[i].grid(True, which="both", axis="y", linestyle=":", alpha=0.5)

ax[0].set_ylabel("RMSE of X")
ax[0].set_title("OI on Different Scales", fontsize=14, fontweight='bold')    

# 把 Legend 移到圖表右側外面，字體放大，投影幕才看得見
ax[0].legend(fontsize=12, loc='center left', bbox_to_anchor=(1.05, 0.5), frameon=False)

ax[1].set_ylabel("RMSE of Y")
ax[2].set_ylabel("RMSE of Z")
ax[2].set_xlabel("Time (MTU)")
ax[2].set_xticks([0, 5, 10, 15, 20])

# plt.subplots_adjust(right=0.85) # 預留空間給右邊的 Legend
plt.savefig(fig_path / "rmse" / "across_scale" / "OI.png", dpi=300, bbox_inches="tight")
plt.close(fig)

##### Across scale 3DVar

In [37]:
# 1. 將圖表拉寬，讓 X 軸的時間展開，減少線條在垂直方向的擁擠感
fig, ax = plt.subplots(3, figsize=(10, 8), sharex="col") 

for i in range(3):
    # 畫 Free Run (黑色虛線墊底)
    ax[i].semilogy(
        time, free_rmse[i][:time_lim][::skips],
        label="Free Run", **style_dict["free"]
    )
    
    # 畫出各種同化設定
    for (key, value) in n3DVar_rmse.items():
        # 如果是該子圖的主角（例如在 i=0 的圖畫 on_X），可以讓它稍微粗一點，以突顯 Downscale control
        lw = 3.0 if (i==0 and key=="on_X") or (i==1 and key=="on_Y") or (i==2 and key=="on_Z") else style_dict[key]["linewidth"]
        
        ax[i].semilogy(
            time, value[i][:time_lim][::skips],
            label=key.replace("_", " "), # 把底線換成空格比較好看
            color=style_dict[key]["color"],
            linestyle=style_dict[key]["linestyle"],
            linewidth=lw,
            alpha=style_dict[key]["alpha"],
            zorder=style_dict[key]["zorder"]
        )

    ax[i].set_xlim(0, 20)
    # 加上網格線，幫助投影幕上對齊數值
    ax[i].grid(True, which="both", axis="y", linestyle=":", alpha=0.5)

ax[0].set_ylabel("RMSE of X")
ax[0].set_title("3D-Var on Different Scales", fontsize=14, fontweight='bold')    

# 把 Legend 移到圖表右側外面，字體放大，投影幕才看得見
ax[0].legend(fontsize=12, loc='center left', bbox_to_anchor=(1.05, 0.5), frameon=False)

ax[1].set_ylabel("RMSE of Y")
ax[2].set_ylabel("RMSE of Z")
ax[2].set_xlabel("Time (MTU)")
ax[2].set_xticks([0, 5, 10, 15, 20])

# plt.subplots_adjust(right=0.85) # 預留空間給右邊的 Legend
plt.savefig(fig_path / "rmse" / "across_scale" / "n3DVar.png", dpi=300, bbox_inches="tight")
plt.close(fig)

##### Across scale incremental 3DVar

In [38]:
# 1. 將圖表拉寬，讓 X 軸的時間展開，減少線條在垂直方向的擁擠感
fig, ax = plt.subplots(3, figsize=(10, 8), sharex="col") 

for i in range(3):
    # 畫 Free Run (黑色虛線墊底)
    ax[i].semilogy(
        time, free_rmse[i][:time_lim][::skips],
        label="Free Run", **style_dict["free"]
    )
    
    # 畫出各種同化設定
    for (key, value) in i3DVar_rmse.items():
        # 如果是該子圖的主角（例如在 i=0 的圖畫 on_X），可以讓它稍微粗一點，以突顯 Downscale control
        lw = 3.0 if (i==0 and key=="on_X") or (i==1 and key=="on_Y") or (i==2 and key=="on_Z") else style_dict[key]["linewidth"]
        
        ax[i].semilogy(
            time, value[i][:time_lim][::skips],
            label=key.replace("_", " "), # 把底線換成空格比較好看
            color=style_dict[key]["color"],
            linestyle=style_dict[key]["linestyle"],
            linewidth=lw,
            alpha=style_dict[key]["alpha"],
            zorder=style_dict[key]["zorder"]
        )

    ax[i].set_xlim(0, 20)
    # 加上網格線，幫助投影幕上對齊數值
    ax[i].grid(True, which="both", axis="y", linestyle=":", alpha=0.5)

ax[0].set_ylabel("RMSE of X")
ax[0].set_title("incremental 3D-Var on Different Scales", fontsize=14, fontweight='bold')    

# 把 Legend 移到圖表右側外面，字體放大，投影幕才看得見
ax[0].legend(fontsize=12, loc='center left', bbox_to_anchor=(1.05, 0.5), frameon=False)

ax[1].set_ylabel("RMSE of Y")
ax[2].set_ylabel("RMSE of Z")
ax[2].set_xlabel("Time (MTU)")
ax[2].set_xticks([0, 5, 10, 15, 20])

# plt.subplots_adjust(right=0.85) # 預留空間給右邊的 Legend
plt.savefig(fig_path / "rmse" / "across_scale" / "i3DVar.png", dpi=300, bbox_inches="tight")
plt.close(fig)

##### Across schemes

In [39]:
fig, ax = plt.subplots(3, figsize=(8, 9), sharex="col")


for i in range(3):

    ax[i].semilogy(time, OI_rmse["on_XYZ"][i][:time_lim][::skips], label="OI")
    ax[i].semilogy(time, n3DVar_rmse["on_XYZ"][i][:time_lim][::skips], label="3DVar")
    ax[i].semilogy(time, i3DVar_rmse["on_XYZ"][i][:time_lim][::skips], label="incremental 3DVar")

ax[i].set_xlim(0, 20)

ax[0].set_ylabel("X")
ax[0].set_title("Comparison of Different Shemes (on X-Y-Z)")    
ax[0].legend()

ax[1].set_ylabel("Y")

ax[2].set_xticks([0, 5, 10, 15, 20])
ax[2].set_ylabel("Z")
ax[2].set_xlabel("Time")

plt.savefig(fig_path / "rmse" / "across_scheme.png", dpi=300, bbox_inches="tight")
plt.close(fig)

#### Visualize ACC

##### Across different OI scale

In [40]:
# 1. 將圖表拉寬，讓 X 軸的時間展開，減少線條在垂直方向的擁擠感
fig, ax = plt.subplots(3, figsize=(10, 8), sharex="col") 


for i in range(3):
    # 畫 Free Run (黑色虛線墊底)
    ax[i].semilogy(
        time, free_acc[i][:time_lim][::skips],
        label="Free Run", **style_dict["free"]
    )
    
    # 畫出各種同化設定
    for (key, value) in OI_acc.items():
        # 如果是該子圖的主角（例如在 i=0 的圖畫 on_X），可以讓它稍微粗一點，以突顯 Downscale control
        lw = 3.0 if (i==0 and key=="on_X") or (i==1 and key=="on_Y") or (i==2 and key=="on_Z") else style_dict[key]["linewidth"]
        
        ax[i].semilogy(
            time, value[i][:time_lim][::skips],
            label=key.replace("_", " "), # 把底線換成空格比較好看
            color=style_dict[key]["color"],
            linestyle=style_dict[key]["linestyle"],
            linewidth=lw,
            alpha=style_dict[key]["alpha"],
            zorder=style_dict[key]["zorder"]
        )

    ax[i].set_xlim(0, 20)
    # 加上網格線，幫助投影幕上對齊數值
    ax[i].grid(True, which="both", axis="y", linestyle=":", alpha=0.5)

ax[0].set_ylabel("ACC of X")
ax[0].set_title("OI on Different Scales", fontsize=14, fontweight='bold')    

# 把 Legend 移到圖表右側外面，字體放大，投影幕才看得見
ax[0].legend(fontsize=12, loc='center left', bbox_to_anchor=(1.05, 0.5), frameon=False)

ax[1].set_ylabel("ACC of Y")
ax[2].set_ylabel("ACC of Z")
ax[2].set_xlabel("Time (MTU)")
ax[2].set_xticks([0, 5, 10, 15, 20])

# plt.subplots_adjust(right=0.85) # 預留空間給右邊的 Legend
plt.savefig(fig_path / "acc" / "across_scale" / "OI.png", dpi=300, bbox_inches="tight")
plt.close(fig)

##### Across different 3DVar scale

In [41]:
# 1. 將圖表拉寬，讓 X 軸的時間展開，減少線條在垂直方向的擁擠感
fig, ax = plt.subplots(3, figsize=(10, 8), sharex="col") 


for i in range(3):
    # 畫 Free Run (黑色虛線墊底)
    ax[i].semilogy(
        time, free_acc[i][:time_lim][::skips],
        label="Free Run", **style_dict["free"]
    )
    
    # 畫出各種同化設定
    for (key, value) in n3DVar_acc.items():
        # 如果是該子圖的主角（例如在 i=0 的圖畫 on_X），可以讓它稍微粗一點，以突顯 Downscale control
        lw = 3.0 if (i==0 and key=="on_X") or (i==1 and key=="on_Y") or (i==2 and key=="on_Z") else style_dict[key]["linewidth"]
        
        ax[i].semilogy(
            time, value[i][:time_lim][::skips],
            label=key.replace("_", " "), # 把底線換成空格比較好看
            color=style_dict[key]["color"],
            linestyle=style_dict[key]["linestyle"],
            linewidth=lw,
            alpha=style_dict[key]["alpha"],
            zorder=style_dict[key]["zorder"]
        )

    ax[i].set_xlim(0, 20)
    # 加上網格線，幫助投影幕上對齊數值
    ax[i].grid(True, which="both", axis="y", linestyle=":", alpha=0.5)

ax[0].set_ylabel("ACC of X")
ax[0].set_title("3D Var on Different Scales", fontsize=14, fontweight='bold')    

# 把 Legend 移到圖表右側外面，字體放大，投影幕才看得見
ax[0].legend(fontsize=12, loc='center left', bbox_to_anchor=(1.05, 0.5), frameon=False)

ax[1].set_ylabel("ACC of Y")
ax[2].set_ylabel("ACC of Z")
ax[2].set_xlabel("Time (MTU)")
ax[2].set_xticks([0, 5, 10, 15, 20])

# plt.subplots_adjust(right=0.85) # 預留空間給右邊的 Legend
plt.savefig(fig_path / "acc" / "across_scale" / "n3DVar.png", dpi=300, bbox_inches="tight")
plt.close(fig)

##### Across scale incremental 3DVar

In [42]:
# 1. 將圖表拉寬，讓 X 軸的時間展開，減少線條在垂直方向的擁擠感
fig, ax = plt.subplots(3, figsize=(10, 8), sharex="col") 


for i in range(3):
    # 畫 Free Run (黑色虛線墊底)
    ax[i].semilogy(
        time, free_acc[i][:time_lim][::skips],
        label="Free Run", **style_dict["free"]
    )
    
    # 畫出各種同化設定
    for (key, value) in i3DVar_acc.items():
        # 如果是該子圖的主角（例如在 i=0 的圖畫 on_X），可以讓它稍微粗一點，以突顯 Downscale control
        lw = 3.0 if (i==0 and key=="on_X") or (i==1 and key=="on_Y") or (i==2 and key=="on_Z") else style_dict[key]["linewidth"]
        
        ax[i].semilogy(
            time, value[i][:time_lim][::skips],
            label=key.replace("_", " "), # 把底線換成空格比較好看
            color=style_dict[key]["color"],
            linestyle=style_dict[key]["linestyle"],
            linewidth=lw,
            alpha=style_dict[key]["alpha"],
            zorder=style_dict[key]["zorder"]
        )

    ax[i].set_xlim(0, 20)
    # 加上網格線，幫助投影幕上對齊數值
    ax[i].grid(True, which="both", axis="y", linestyle=":", alpha=0.5)

ax[0].set_ylabel("ACC of X")
ax[0].set_title("incremental 3D-Var on Different Scales", fontsize=14, fontweight='bold')    

# 把 Legend 移到圖表右側外面，字體放大，投影幕才看得見
ax[0].legend(fontsize=12, loc='center left', bbox_to_anchor=(1.05, 0.5), frameon=False)

ax[1].set_ylabel("ACC of Y")
ax[2].set_ylabel("ACC of Z")
ax[2].set_xlabel("Time (MTU)")
ax[2].set_xticks([0, 5, 10, 15, 20])

# plt.subplots_adjust(right=0.85) # 預留空間給右邊的 Legend
plt.savefig(fig_path / "acc" / "across_scale" / "i3DVar.png", dpi=300, bbox_inches="tight")
plt.close(fig)

##### Across schemes

In [43]:
fig, ax = plt.subplots(3, figsize=(8, 9), sharex="col")


for i in range(3):

    ax[i].plot(time, OI_acc["on_XYZ"][i][:time_lim][::skips], label="OI")
    ax[i].plot(time, n3DVar_acc["on_XYZ"][i][:time_lim][::skips], label="3DVar")
    ax[i].plot(time, i3DVar_acc["on_XYZ"][i][:time_lim][::skips], label="incremental 3DVar")

ax[i].set_xlim(0, 20)

ax[0].set_ylabel("X")
ax[0].set_title("Comparison of Different Shemes (on X-Y-Z)")    
ax[0].legend()

ax[1].set_ylabel("Y")

ax[2].set_xticks([0, 5, 10, 15, 20])
ax[2].set_ylabel("Z")
ax[2].set_xlabel("Time")

plt.savefig(fig_path / "acc" / "across_scheme.png", dpi=300, bbox_inches="tight")
plt.close(fig)